In [1]:
import os
import time
import json
from pathlib import Path
import pandas as pd
import glob
import numpy as np
import pprint
import requests
from datetime import datetime

In [2]:
df = pd.read_csv("combined_votes_cleaned.csv")

In [105]:
#Let's start by figuring out how often the council or the EP wins a second reading overall
df["vote_id"] = df["vote_id"].str.replace("eli/dl/event/MTG-PL-", "")
df = df.sort_values("vote_id")
df.head()

,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss,day_category
63,2014-01-15-VOT-ITM-344699-7,NaN,NaN,Roll Call / Electronic,Rejected,NaN,57.0,604.0,NaN,eli/dl/doc/PV-7-2014-01-15-VOT,7.0,17.0,Council win,Wednesday
64,2014-03-11-VOT-ITM-348809-9,NaN,NaN,Roll Call / Electronic,Rejected,NaN,40.0,624.0,NaN,eli/dl/doc/PV-7-2014-03-11-VOT,9.0,12.0,Council win,Tuesday
65,2014-04-02-VOT-ITM-350909-6,NaN,NaN,Auto-adopted,Approved,NaN,NaN,NaN,NaN,eli/dl/doc/PV-7-2014-04-02-VOT,6.0,NaN,Council win,Wednesday
37,2014-04-03-VOT-ITM-350903-1,NaN,NaN,Auto-adopted,Approved,NaN,NaN,NaN,NaN,eli/dl/doc/PV-7-2014-04-03-VOT,1.0,NaN,Council win,Thursday
67,2014-04-15-VOT-ITM-352267-11,NaN,NaN,Auto-adopted,Approved,NaN,NaN,NaN,NaN,eli/dl/doc/PV-7-2014-04-15-VOT,11.0,NaN,Council win,Tuesday


In [106]:
df.to_csv("combined_votes_cleaned.csv", index=False)

In [16]:
wins = df["Council win or loss"].value_counts()["Council win"]
print(f"In our 96 vote sample the EU council wins {wins} times, or {round(wins/len(df)*100, 1)}% of the time.")

In our 96 vote sample the EU council wins 91 times, or 94.8% of the time.


In [83]:
df[df["Council win or loss"] == "Council loss"]

,Unnamed: 0.1,Unnamed: 0,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss,day_category
4,4,4,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195778,2026-07-09T13:22:50+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,611.0,369.0,236.0,Draft legislative act,NaN,NaN,NaN,Council loss,Thursday
13,13,13,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195787,2026-07-09T13:25:49+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,602.0,362.0,235.0,Draft legislative act,NaN,NaN,NaN,Council loss,Thursday
56,56,56,eli/dl/event/MTG-PL-2026-01-21-VOT-ITM-981675,eli/dl/event/MTG-PL-2026-01-21-DEC-183557,NaN,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,656.0,632.0,15.0,NaN,NaN,NaN,NaN,Council loss,Wednesday
60,60,60,eli/dl/event/MTG-PL-2026-07-07-VOT-ITM-992933,eli/dl/event/MTG-PL-2026-07-07-DEC-195007,NaN,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,661.0,646.0,12.0,NaN,NaN,NaN,NaN,Council loss,Tuesday
71,71,71,eli/dl/event/MTG-PL-2015-01-13-VOT-ITM-398848-5,NaN,NaN,Roll Call / Electronic,Adopted,NaN,480.0,159.0,NaN,eli/dl/doc/PV-8-2015-01-13-VOT,5.0,58.0,Council loss,Tuesday


In [61]:
mask = (361 > df['votes_favor']) & (df['votes_favor'] > df['votes_against'])
df_simple_maj = df[mask].copy()

print(df_simple_maj)

    Unnamed: 0.1  Unnamed: 0                                        vote_id  \
1              1           1  eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015   
9              9           9  eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015   
12            12          12  eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015   
14            14          14  eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015   
22            22          22  eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015   

                                  decision_id                 start_date  \
1   eli/dl/event/MTG-PL-2026-07-09-DEC-195775  2026-07-09T13:21:39+02:00   
9   eli/dl/event/MTG-PL-2026-07-09-DEC-195783  2026-07-09T13:24:23+02:00   
12  eli/dl/event/MTG-PL-2026-07-09-DEC-195786  2026-07-09T13:25:26+02:00   
14  eli/dl/event/MTG-PL-2026-07-09-DEC-195788  2026-07-09T13:26:11+02:00   
22  eli/dl/event/MTG-PL-2026-07-09-DEC-195796  2026-07-09T13:28:06+02:00   

                                              method  \
1   def/ep-d

In [29]:
print(f"Out of all these second readings there were only {len(df_simple_maj)} votes that would have won if a simple majority was the requirement")
print(f"However, all of these were votes on Chat control, taken on the last thursday before summer recess")
print(f"The average attendance for these votes was {round(df_simple_maj["attendees"].mean())} voters.")
print(f"That is {720-(round(df_simple_maj["attendees"].mean()))} fewer than the number of seats.")

Out of all these second readings there were only 5 votes that would have won if a simple majority was the requirement
However, all of these were votes on Chat control, taken on the last thursday before summer recess
The average attendance for these votes was 607 voters.
That is 113 fewer than the number of seats.


In [33]:
tues_wins = df[df["day_category"] == "Tuesday"]["Council win or loss"].value_counts()["Council win"]
wedns_wins = df[df["day_category"] == "Wednesday"]["Council win or loss"].value_counts()["Council win"]
thurs_wins = df[df["day_category"] == "Thursday"]["Council win or loss"].value_counts()["Council win"]

print(f"On Tuesday 2nd reading votes, the council won {round(tues_wins/len(df[df["day_category"] == "Tuesday"])*100, 1)}% of the times")
print(f"On Wednesday 2nd reading votes, the council won {round(wedns_wins/len(df[df["day_category"] == "Wednesday"])*100, 1)}% of the times")
print(f"On Thursday 2nd reading votes, the council won {round(thurs_wins/len(df[df["day_category"] == "Thursday"])*100, 1)}% of the times")

On Tuesday 2nd reading votes, the council won 93.8% of the times
On Wednesday 2nd reading votes, the council won 95.0% of the times
On Thursday 2nd reading votes, the council won 95.5% of the times


In [49]:
#How many of these second readings came to actual votes and how many were auto-adopted
actual_votes = df[df["method"] != "Auto-adopted"]
print(len(df))
print(f"Of these {len(df)} votes many never actually came to counting, but were adopted without voting on it. Only {len(actual_votes)} came to real votes.")

96
Of these 96 votes many never actually came to counting, but were adopted without voting on it. Only 56 came to real votes.


In [34]:
df_all = pd.read_csv("all_meetings_raw.csv")
print(len(df_all))
df_all.head()

215


,Unnamed: 0,id,type,activity_date,activity_end_date,activity_id,activity_label,activity_start_date,consists_of,documented_by_a_realization_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in
0,0,eli/dl/event/MTG-PL-2024-01-15,Activity,2024-01-15,2024-01-15T23:00:00+01:00,MTG-PL-2024-01-15,"{'hu': '2024. január 15., hétfő', 'sv': 'Månda...",2024-01-15T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-15-PVCRE-ITM-14'...,['eli/dl/doc/OJQ-9-2024-01-15'],def/ep-activities/PLENARY_SITTING,"['person/197683', 'person/197579', 'person/125...","['person/99283', 'person/88882', 'person/12480...",581.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-15-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
1,1,eli/dl/event/MTG-PL-2024-01-16,Activity,2024-01-16,2024-01-16T23:00:00+01:00,MTG-PL-2024-01-16,"{'da': 'Tirsdag den 16. januar 2024', 'mt': ""I...",2024-01-16T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-16-PVCRE-ITM-11'...,['eli/dl/doc/OJQ-9-2024-01-16'],def/ep-activities/PLENARY_SITTING,"['person/36392', 'person/125030', 'person/1974...","['person/125214', 'person/197754', 'person/967...",636.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-16-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
2,2,eli/dl/event/MTG-PL-2024-01-17,Activity,2024-01-17,2024-01-17T23:00:00+01:00,MTG-PL-2024-01-17,"{'it': 'Mercoledì 17 gennaio 2024', 'en': 'Wed...",2024-01-17T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-17-VOT-ITM-93994...,['eli/dl/doc/OJQ-9-2024-01-17'],def/ep-activities/PLENARY_SITTING,"['person/198183', 'person/36392', 'person/9677...","['person/197607', 'person/197770', 'person/125...",637.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-17-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
3,3,eli/dl/event/MTG-PL-2024-01-18,Activity,2024-01-18,2024-01-18T23:00:00+01:00,MTG-PL-2024-01-18,"{'lt': 'Ketvirtadienis, 2024 m. sausio 18 d.',...",2024-01-18T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-93993...,['eli/dl/doc/OJQ-9-2024-01-18'],def/ep-activities/PLENARY_SITTING,"['person/197683', 'person/197478', 'person/132...","['person/4746', 'person/197772', 'person/24457...",586.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-18-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
4,4,eli/dl/event/MTG-PL-2024-01-25,Activity,2024-01-25,2024-01-25T23:00:00+01:00,MTG-PL-2024-01-25,"{'de': 'Donnerstag, 25. Januar 2024', 'it': 'G...",2024-01-25T01:00:00+01:00,"['eli/dl/event/MTG-PL-2024-01-25-PVCRE-ITM-5',...",['eli/dl/doc/OJQ-9-2024-01-25'],def/ep-activities/PLENARY_SITTING,"['person/197579', 'person/124770', 'person/197...","['person/197840', 'person/197782', 'person/197...",340.0,org/ep-9,"['eli/dl/doc/CRE-9-2024-01-25', 'eli/dl/doc/PV...",http://publications.europa.eu/resource/authori...,NaN


In [35]:
df_all.sort_values("activity_date")

,Unnamed: 0,id,type,activity_date,activity_end_date,activity_id,activity_label,activity_start_date,consists_of,documented_by_a_realization_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in
157,157,eli/dl/event/MTG-PL-2023-01-16,Activity,2023-01-16,2023-01-16T23:00:00+01:00,MTG-PL-2023-01-16,"{'lt': 'Pirmadienis, 2023 m. sausio 16 d.', 'p...",2023-01-16T01:00:00+01:00,['eli/dl/event/MTG-PL-2023-01-16-PVCRE-ITM-15'...,['eli/dl/doc/OJQ-9-2023-01-16'],def/ep-activities/PLENARY_SITTING,"['person/197518', 'person/197398', 'person/182...","['person/197770', 'person/125023', 'person/197...",601.0,org/ep-9,"['eli/dl/doc/PV-9-2023-01-16', 'eli/dl/doc/PV-...",http://publications.europa.eu/resource/authori...,NaN
158,158,eli/dl/event/MTG-PL-2023-01-17,Activity,2023-01-17,2023-01-17T23:00:00+01:00,MTG-PL-2023-01-17,"{'es': 'Martes 17 de enero de 2023', 'fi': 'Ti...",2023-01-17T01:00:00+01:00,"['eli/dl/event/MTG-PL-2023-01-17-PVCRE-ITM-7',...",['eli/dl/doc/OJQ-9-2023-01-17'],def/ep-activities/PLENARY_SITTING,"['person/197398', 'person/197827', 'person/206...","['person/197538', 'person/197572', 'person/125...",642.0,org/ep-9,"['eli/dl/doc/CRE-9-2023-01-17', 'eli/dl/doc/PV...",http://publications.europa.eu/resource/authori...,NaN
159,159,eli/dl/event/MTG-PL-2023-01-18,Activity,2023-01-18,2023-01-18T23:00:00+01:00,MTG-PL-2023-01-18,"{'da': 'Onsdag den 18. januar 2023', 'sl': 'Sr...",2023-01-18T01:00:00+01:00,['eli/dl/event/MTG-PL-2023-01-18-PVCRE-ITM-12'...,['eli/dl/doc/OJQ-9-2023-01-18'],def/ep-activities/PLENARY_SITTING,"['person/96698', 'person/124734', 'person/1829...","['person/197624', 'person/197663', 'person/197...",644.0,org/ep-9,"['eli/dl/doc/PV-9-2023-01-18-RCV', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
160,160,eli/dl/event/MTG-PL-2023-01-19,Activity,2023-01-19,2023-01-19T23:00:00+01:00,MTG-PL-2023-01-19,"{'fr': 'Jeudi 19 janvier 2023', 'lt': 'Ketvirt...",2023-01-19T01:00:00+01:00,"['eli/dl/event/MTG-PL-2023-01-19-PVCRE-ITM-7',...",['eli/dl/doc/OJQ-9-2023-01-19'],def/ep-activities/PLENARY_SITTING,"['person/182995', 'person/197547', 'person/197...","['person/197462', 'person/96713', 'person/1978...",563.0,org/ep-9,"['eli/dl/doc/PV-9-2023-01-19-RCV', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
161,161,eli/dl/event/MTG-PL-2023-01-26,Activity,2023-01-26,2023-01-26T23:00:00+01:00,MTG-PL-2023-01-26,"{'et': 'Neljapäev, 26. jaanuar 2023', 'pt': 'Q...",2023-01-26T01:00:00+01:00,"['eli/dl/event/MTG-PL-2023-01-26-PVCRE-ITM-6',...",['eli/dl/doc/OJQ-9-2023-01-26'],def/ep-activities/PLENARY_SITTING,"['person/197824', 'person/96698', 'person/1973...","['person/33982', 'person/126644', 'person/1252...",342.0,org/ep-9,"['eli/dl/doc/PV-9-2023-01-26', 'eli/dl/doc/CRE...",http://publications.europa.eu/resource/authori...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,152,eli/dl/event/MTG-PL-2026-11-26,Activity,2026-11-26,2026-11-26T23:00:00+01:00,MTG-PL-2026-11-26,"{'hu': '2026. november 26., csütörtök', 'lv': ...",2026-11-26T01:00:00+01:00,NaN,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-10,NaN,http://publications.europa.eu/resource/authori...,NaN
153,153,eli/dl/event/MTG-PL-2026-12-14,Activity,2026-12-14,2026-12-14T23:00:00+01:00,MTG-PL-2026-12-14,"{'fi': 'Maanantai 14. joulukuuta 2026', 'et': ...",2026-12-14T01:00:00+01:00,NaN,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-10,NaN,http://publications.europa.eu/resource/authori...,NaN
154,154,eli/dl/event/MTG-PL-2026-12-15,Activity,2026-12-15,2026-12-15T23:00:00+01:00,MTG-PL-2026-12-15,"{'fi': 'Tiistai 15. joulukuuta 2026', 'bg': 'В...",2026-12-15T01:00:00+01:00,NaN,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-10,NaN,http://publications.europa.eu/resource/authori...,NaN
155,155,eli/dl/event/MTG-PL-2026-12-16,Activity,2026-12-16,2026-12-16T23:00:00+01:00,MTG-PL-2026-12-

In [36]:
#We will sort by number of attendees to find when the fewest MEPs are in parliament
#We will use this data set instead of the full 12 years because during these years the data is consistent
df_all.sort_values("number_of_attendees")

,Unnamed: 0,id,type,activity_date,activity_end_date,activity_id,activity_label,activity_start_date,consists_of,documented_by_a_realization_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in
26,26,eli/dl/event/MTG-PL-2024-07-19,Activity,2024-07-19,2024-07-19T23:00:00+02:00,MTG-PL-2024-07-19,"{'lt': 'Penktadienis, 2024 m. liepos 19 d.', '...",2024-07-19T01:00:00+02:00,"['eli/dl/event/MTG-PL-2024-07-19-PVCRE-ITM-8',...",['eli/dl/doc/OJQ-10-2024-07-19'],def/ep-activities/PLENARY_SITTING,"['person/122978', 'person/256820']","['person/256905', 'person/124726', 'person/256...",333.0,org/ep-10,"['eli/dl/doc/CRE-10-2024-07-19', 'eli/dl/doc/P...",http://publications.europa.eu/resource/authori...,NaN
4,4,eli/dl/event/MTG-PL-2024-01-25,Activity,2024-01-25,2024-01-25T23:00:00+01:00,MTG-PL-2024-01-25,"{'de': 'Donnerstag, 25. Januar 2024', 'it': 'G...",2024-01-25T01:00:00+01:00,"['eli/dl/event/MTG-PL-2024-01-25-PVCRE-ITM-5',...",['eli/dl/doc/OJQ-9-2024-01-25'],def/ep-activities/PLENARY_SITTING,"['person/197579', 'person/124770', 'person/197...","['person/197840', 'person/197782', 'person/197...",340.0,org/ep-9,"['eli/dl/doc/CRE-9-2024-01-25', 'eli/dl/doc/PV...",http://publications.europa.eu/resource/authori...,NaN
161,161,eli/dl/event/MTG-PL-2023-01-26,Activity,2023-01-26,2023-01-26T23:00:00+01:00,MTG-PL-2023-01-26,"{'et': 'Neljapäev, 26. jaanuar 2023', 'pt': 'Q...",2023-01-26T01:00:00+01:00,"['eli/dl/event/MTG-PL-2023-01-26-PVCRE-ITM-6',...",['eli/dl/doc/OJQ-9-2023-01-26'],def/ep-activities/PLENARY_SITTING,"['person/197824', 'person/96698', 'person/1973...","['person/33982', 'person/126644', 'person/1252...",342.0,org/ep-9,"['eli/dl/doc/PV-9-2023-01-26', 'eli/dl/doc/CRE...",http://publications.europa.eu/resource/authori...,NaN
164,164,eli/dl/event/MTG-PL-2023-02-09,Activity,2023-02-09,2023-02-09T23:00:00+01:00,MTG-PL-2023-02-09,"{'pl': 'Czwartek, 9 lutego 2023', 'mt': ""Il-Ħa...",2023-02-09T01:00:00+01:00,"['eli/dl/event/MTG-PL-2023-02-09-PVCRE-ITM-7',...",['eli/dl/doc/OJQ-9-2023-02-09'],def/ep-activities/PLENARY_SITTING,"['person/124734', 'person/197827', 'person/124...","['person/197694', 'person/125043', 'person/197...",381.0,org/ep-9,"['eli/dl/doc/PV-9-2023-02-09-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
41,41,eli/dl/event/MTG-PL-2024-11-19,Activity,2024-11-19,2024-11-19T23:00:00+01:00,MTG-PL-2024-11-19,"{'it': 'Martedì 19 novembre 2024', 'el': 'Τρίτ...",2024-11-19T01:00:00+01:00,"['eli/dl/event/MTG-PL-2024-11-19-PVCRE-ITM-4',...",['eli/dl/doc/OJQ-10-2024-11-19'],def/ep-activities/PLENARY_SITTING,['person/257130'],"['person/257113', 'person/257037', 'person/197...",394.0,org/ep-10,"['eli/dl/doc/PV-10-2024-11-19', 'eli/dl/doc/CR...",http://publications.europa.eu/resource/authori...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,152,eli/dl/event/MTG-PL-2026-11-26,Activity,2026-11-26,2026-11-26T23:00:00+01:00,MTG-PL-2026-11-26,"{'hu': '2026. november 26., csütörtök', 'lv': ...",2026-11-26T01:00:00+01:00,NaN,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-10,NaN,http://publications.europa.eu/resource/authori...,NaN
153,153,eli/dl/event/MTG-PL-2026-12-14,Activity,2026-12-14,2026-12-14T23:00:00+01:00,MTG-PL-2026-12-14,"{'fi': 'Maanantai 14. joulukuuta 2026', 'et': ...",2026-12-14T01:00:00+01:00,NaN,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-10,NaN,http://publications.europa.eu/resource/authori...,NaN
154,154,eli/dl/event/MTG-PL-2026-12-15,Activity,2026-12-15,2026-12-15T23:00:00+01:00,MTG-PL-2026-12-15,"{'fi': 'Tiistai 15. joulukuuta 2026', 'bg': 'В...",2026-12-15T01:00:00+01:00,NaN,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-10,NaN,http://publications.europa.eu/resource/authori...,NaN
155,155,eli/dl/event/MTG-PL-2026-12-16,Activity,2026-12-16,2026-12-16T23:00:00+01:00,MTG-PL-2026-12-16,"{'et': 'Kolmapäev, 16. detsember 2026', 'de'

In [40]:
print(f"Of the 215 plenary days in the set, the one with the worst attendance was also a final session before the summer recess, although on a friday, on July 19 2024, with 333 attendees.")

Of the 215 plenary days in the set, the one with the worst attendance was also a final session before the summer recess, although on a friday, on July 19 2024, with 333 attendees.


In [42]:
df_lowest_attendance = df_all.sort_values("number_of_attendees").head(50)
dates = pd.to_datetime(df_lowest_attendance["activity_date"], errors="coerce")
weekday_distribution = dates.dt.day_name().value_counts()
print(weekday_distribution)

activity_date
Thursday     23
Monday       14
Wednesday     9
Tuesday       3
Friday        1
Name: count, dtype: int64


In [43]:
print("Thursdays have by far the lowest attendance, with nearly half (23) of the 50 worst attended days being Thursdays.")

Thursdays have by far the lowest attendance, with nearly half (23) of the 50 worst attended days being Thursdays.


In [50]:
df_all["activity_date"] = pd.to_datetime(df_all["activity_date"], errors="coerce")
df_all["day_of_week"] = df_all["activity_date"].dt.day_name()

In [110]:
#Average attendance per weekday
print(df_all.groupby("day_of_week")["number_of_attendees"].mean())
thurs_avrg = df_all.groupby("day_of_week")["number_of_attendees"].mean()["Thursday"]
tues_avrg = df_all.groupby("day_of_week")["number_of_attendees"].mean()["Tuesday"]
wedns_avrg = df_all.groupby("day_of_week")["number_of_attendees"].mean()["Wednesday"]
mon_avrg = df_all.groupby("day_of_week")["number_of_attendees"].mean()["Monday"]
print(mon_avrg, tues_avrg, wedns_avrg, thurs_avrg)

day_of_week
Friday       333.000000
Monday       597.634146
Thursday     586.888889
Tuesday      640.733333
Wednesday    632.096154
Name: number_of_attendees, dtype: float64
597.6341463414634 640.7333333333333 632.0961538461538 586.8888888888889


In [58]:
print("Excluding fridays, of which there was only one, thursdays have the worst attendance, followed by mondays, on which no major votes take place.")
print(f"On average there are only {round(thurs_avrg)} people in attandance on Thursdays.")
print(f"Thats {round(tues_avrg-thurs_avrg)} fewer than on tuesdays, the day with best attendance.")

Excluding fridays, of which there was only one, thursdays have the worst attendance, followed by mondays, on which no major votes take place.
On average there are only 587 people in attandance on Thursdays.
Thats 54 fewer than on tuesdays, the day with best attendance.


In [59]:
df_all_dec = pd.read_csv("all_days_vote_results_w_time_stamps.csv")

In [65]:
df_all_dec

,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,absolute_majority
0,eli/dl/event/MTG-PL-2023-11-08-VOT-ITM-000001,eli/dl/event/MTG-PL-2023-11-08-DEC-160016,2023-11-08T15:50:48+01:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,Unknown,NaN,377.0,24.0,NaN,False
1,eli/dl/event/MTG-PL-2026-02-11-VOT-ITM-983762,eli/dl/event/MTG-PL-2026-02-11-DEC-184692,2026-02-11T12:27:41+01:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,606.0,428.0,156.0,Motion for a resolution B10-0108/2026\n(ENVI C...,False
2,eli/dl/event/MTG-PL-2026-03-12-VOT-ITM-986076,eli/dl/event/MTG-PL-2026-03-12-DEC-186149,NaN,def/ep-decision-methods/VOTE_HAND,def/ep-statuses/REJECTED,NaN,NaN,NaN,NaN,False
3,eli/dl/event/MTG-PL-2025-06-19-VOT-ITM-965710,NaN,NaN,Unknown,Unknown,NaN,NaN,NaN,NaN,False
4,eli/dl/event/MTG-PL-2025-10-21-VOT-ITM-974693,eli/dl/event/MTG-PL-2025-10-21-DEC-179820,2025-10-21T13:21:25+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,644.0,533.0,43.0,Provisional agreement,False
...,...,...,...,...,...,...,...,...,...,...
1817,eli/dl/event/MTG-PL-2024-04-24-VOT-ITM-946086,eli/dl/event/MTG-PL-2024-04-24-DEC-168483,2024-04-24T18:25:05+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,595.0,582.0,10.0,Provisional agreement,False
1818,eli/dl/event/MTG-PL-2024-01-16-VOT-ITM-939922,eli/dl/event/MTG-PL-2024-01-16-DEC-163074,2024-01-16T12:04:51+01:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,598.0,518.0,46.0,Provisional agreement,False
1819,eli/dl/event/MTG-PL-2023-11-09-VOT-ITM-939648,NaN,NaN,Unknown,Unknown,NaN,NaN,NaN,NaN,False
1820,eli/dl/event/MTG-PL-2023-02-02-VOT-ITM-938629,NaN,NaN,Unknown,Unknown,NaN,NaN,NaN,NaN,False


In [71]:
#Let's compare the average attendance on thursday afternoons with avrg attendance thursdays overall
print(len(df_all_dec))
df_time_stamps = df_all_dec.dropna(subset=["start_date"])

1822


In [72]:
df_time_stamps["start_date"] = df_time_stamps["start_date"].str[:-6]
df_time_stamps.head()

,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,absolute_majority
0,eli/dl/event/MTG-PL-2023-11-08-VOT-ITM-000001,eli/dl/event/MTG-PL-2023-11-08-DEC-160016,2023-11-08T15:50:48,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,Unknown,NaN,377.0,24.0,NaN,False
1,eli/dl/event/MTG-PL-2026-02-11-VOT-ITM-983762,eli/dl/event/MTG-PL-2026-02-11-DEC-184692,2026-02-11T12:27:41,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,606.0,428.0,156.0,Motion for a resolution B10-0108/2026\n(ENVI C...,False
4,eli/dl/event/MTG-PL-2025-10-21-VOT-ITM-974693,eli/dl/event/MTG-PL-2025-10-21-DEC-179820,2025-10-21T13:21:25,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,644.0,533.0,43.0,Provisional agreement,False
6,eli/dl/event/MTG-PL-2024-04-11-VOT-ITM-944643,eli/dl/event/MTG-PL-2024-04-11-DEC-167156,2024-04-11T13:19:32,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,600.0,527.0,40.0,Motion for a resolution,False
8,eli/dl/event/MTG-PL-2025-07-09-VOT-ITM-965785,eli/dl/event/MTG-PL-2025-07-09-DEC-178102,2025-07-09T13:56:20,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,670.0,423.0,194.0,NaN,False


In [73]:
df_time_stamps["start_date"] = pd.to_datetime(df_time_stamps["start_date"], errors="coerce")

In [74]:
df_time_stamps["day_of_week"] = df_time_stamps["start_date"].dt.day_name()
df_time_stamps["day_of_week"].value_counts()

day_of_week
Wednesday    321
Thursday     206
Tuesday      179
Monday       124
Name: count, dtype: int64

In [75]:
#filter these to only thursdays after lunch
#thursdays
thursdays = df_time_stamps[df_time_stamps["day_of_week"] == "Thursday"].copy()

#after lunch
thurs_post_lunch = thursdays[thursdays["start_date"].dt.hour >= 12]

print(f"Total Thursday votes: {len(thursdays)}")
print(f"Thursday afternoon votes: {len(thurs_post_lunch)}")

Total Thursday votes: 206
Thursday afternoon votes: 179


In [80]:
#average attendees after lunch
round(thurs_post_lunch["attendees"].mean())
print(f"After noon the average attendance number drops even further, to {round(thurs_post_lunch["attendees"].mean())}")
print(f"Thats {round(thurs_avrg - thurs_post_lunch["attendees"].mean())} fewer than the number who signed their names on that morning's attendance call.")

After noon the average attendance number drops even further, to 574
Thats 13 fewer than the number who signed their names on that morning's attendance call.


In [97]:
#How often do normal thursday votes (not 2nd readings) reach absolute majority numbers compared to tuesdays?
total_thursdays = len(df_time_stamps[df_time_stamps["day_of_week"] == "Thursday"])
total_tuesdays = len(df_time_stamps[df_time_stamps["day_of_week"] == "Tuesday"])
total_wednesdays = len(df_time_stamps[df_time_stamps["day_of_week"] == "Wednesday"])

# 2. Get the number of votes > 360 per day
big_thursdays = len(df_time_stamps[(df_time_stamps["day_of_week"] == "Thursday") & (df_time_stamps["votes_favor"] > 360)])
big_tuesdays = len(df_time_stamps[(df_time_stamps["day_of_week"] == "Tuesday") & (df_time_stamps["votes_favor"] > 360)])
big_wednesdays = len(df_time_stamps[(df_time_stamps["day_of_week"] == "Wednesday") & (df_time_stamps["votes_favor"] > 360)])

# 3. Print the clean results with rounded percentages
print(f"On Thursdays: {big_thursdays} of {total_thursdays} ({(big_thursdays/total_thursdays)*100:.1f}%) reached more than 360 votes in favor")
print(f"On Tuesdays: {big_tuesdays} of {total_tuesdays} ({(big_tuesdays/total_tuesdays)*100:.1f}%) reached more than 360 votes in favor")
print(f"On Wednesdays: {big_wednesdays} of {total_wednesdays} ({(big_wednesdays/total_wednesdays)*100:.1f}%) reached more than 360 votes in favor")

On Thursdays: 125 of 206 (60.7%) reached more than 360 votes in favor
On Tuesdays: 147 of 179 (82.1%) reached more than 360 votes in favor
On Wednesdays: 223 of 321 (69.5%) reached more than 360 votes in favor


In [111]:
df_all.to_csv("all_meetings_w_weekdays.csv", index=False)